# Assignment 2: RAG-Enhanced Pricing Agent with Historical Knowledge

## Objective
Add factual grounding to our pricing agent using a **pricing knowledge base** with historical data, category benchmarks, and proven pricing strategies.

## Requirements
**RAG Knowledge Base Includes:**
- Category-level elasticity benchmarks
- Historical margins
- 20-50 sample past pricing decisions
- Competitor trends
- Price recommendation guidelines

**Agent Behavior:**
- Pulls past examples
- Justifies price using retrieved data
- Reduces hallucination compared to Iteration 1

## Setup & Dependencies

Install the required packages for RAG implementation.

In [1]:
# Install required packages for RAG
!pip install -q langchain langchain-groq langchain-community
!pip install -q chromadb sentence-transformers
!pip install -q pandas numpy

In [2]:
# Import required libraries
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_text_splitters import CharacterTextSplitter
import os
import getpass
import pandas as pd
import json
from typing import List, Dict


In [3]:
# Set up your Groq API key
print("Please enter your Groq API key:")
print("(You can get one free at: https://console.groq.com/)")
groq_api_key = getpass.getpass("Groq API Key: ")
os.environ["GROQ_API_KEY"] = groq_api_key
print("API key set successfully!")

Please enter your Groq API key:
(You can get one free at: https://console.groq.com/)
API key set successfully!


In [4]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
)

## Create Pricing Knowledge Base

Create a comprehensive pricing knowledge base with historical data and benchmarks.

In [5]:
# 1. Category-level elasticity benchmarks
elasticity_benchmarks = [
    {"category": "Electronics", "elasticity": "High", "description": "Highly sensitive to price changes, small price drops can lead to large increases in demand."},
    {"category": "Clothing", "elasticity": "Medium", "description": "Moderately sensitive to price changes, consumers may switch brands but still purchase within the category."},
    {"category": "Groceries", "elasticity": "Low", "description": "Less sensitive to price changes, consumers need these products regardless of price fluctuations."},
    {"category": "Furniture", "elasticity": "Medium-High", "description": "Sensitive to price changes, especially for non-essential items, but may have some brand loyalty."}
]

# 2. Historical margin data by category  
historical_margins = [
    {"category": "Electronics", "avg_margin": "12-18%", "description": "Margins can vary widely based on product type, with high-end electronics often commanding higher margins."},
    {"category": "Clothing", "avg_margin": "30-50%", "description": "Fashion items typically have higher margins, especially for premium brands."},
    {"category": "Groceries", "avg_margin": "5-15%", "description": "Margins are generally lower due to high competition and price sensitivity."},
    {"category": "Furniture", "avg_margin": "20-40%", "description": "Margins can be substantial, particularly for custom or high-end pieces."}
]

# 3. Sample past pricing decisions (20-50 examples)
pricing_decisions = [
    {
        "product": "Samsung Galaxy Smartphone",
        "category": "Electronics",
        "cost": 450,
        "recommended_price": 499,
        "margin": "11%",
        "competitor_price": 529,
        "outcome": "Successful - gained 15% market share",
        "reasoning": "Aggressive pricing in high elasticity category drove volume"
    },
    {
        "product": "Levi's Jeans",
        "category": "Clothing",
        "cost": 30,
        "recommended_price": 49.99,
        "margin": "66%",
        "competitor_price": 59.99,
        "outcome": "Successful - maintained market share",
        "reasoning": "Competitive pricing in medium elasticity category retained customers"
    },
    {
        "product": "Organic Milk",
        "category": "Groceries",
        "cost": 3,
        "recommended_price": 3.49,
        "margin": "16%",
        "competitor_price": 3.99,
        "outcome": "Successful - increased sales by 10%",
        "reasoning": "Slightly lower price in low elasticity category attracted price-sensitive customers"
    },
    {
        "product": "IKEA Dining Table",
        "category": "Furniture",
        "cost": 150,
        "recommended_price": 199,
        "margin": "33%",
        "competitor_price": 249,
        "outcome": "Successful - gained market share",
        "reasoning": "Aggressive pricing in medium-high elasticity category drove volume"
    },
    {
        "product": "Apple MacBook Pro",
        "category": "Electronics",
        "cost": 1200,
        "recommended_price": 1299,      
        "margin": "8%",
        "competitor_price": 1399,
        "outcome": "Successful - maintained market share",
        "reasoning": "Premium pricing in high elasticity category maintained brand perception"
    },
    {
        "product": "Nike Running Shoes",
        "category": "Clothing",         
        "cost": 60,
        "recommended_price": 89.99,             
        "margin": "50%",
        "competitor_price": 99.99,
        "outcome": "Successful - increased sales by 20%",
        "reasoning": "Competitive pricing in medium elasticity category attracted customers from competitors"
    },
    {
        "product": "Whole Wheat Bread",         
        "category": "Groceries",
        "cost": 2,                  
        "recommended_price": 2.49,
        "margin": "25%",
        "competitor_price": 2.99,
        "outcome": "Successful - increased sales by 15%",
        "reasoning": "Slightly lower price in low elasticity category attracted price-sensitive customers"
    }   
]

# 4. Price recommendation guidelines
pricing_guidelines = [
    "For high elasticity categories, consider aggressive pricing strategies to drive volume and gain market share.",
    "For medium elasticity categories, maintain competitive pricing to retain customers while ensuring healthy margins.",
    "For low elasticity categories, focus on value proposition and brand loyalty rather than price competition.",
    "Always analyze competitor pricing and market trends before making pricing decisions.",
    "Consider the cost structure and desired margin when setting prices, but also factor in customer perception and willingness to pay."
]

print(f"Created knowledge base with:")
print(f"- {len(elasticity_benchmarks)} category elasticity benchmarks")
print(f"- {len(historical_margins)} historical margin references") 
print(f"- {len(pricing_decisions)} past pricing decisions")
print(f"- {len(pricing_guidelines)} pricing guidelines")

Created knowledge base with:
- 4 category elasticity benchmarks
- 4 historical margin references
- 7 past pricing decisions
- 5 pricing guidelines


## Create Vector Store for RAG

Convert our knowledge base into searchable documents and create embeddings.

In [6]:
def create_pricing_documents() -> List[Document]:
    """Convert pricing knowledge into searchable documents"""
    documents = []
    
    for benchmark in elasticity_benchmarks:
        content = f"Category: {benchmark['category']}\nElasticity: {benchmark['elasticity']}\nDescription: {benchmark['description']}"
        metadata = {"type": "elasticity_benchmark", "category": benchmark["category"]}
        documents.append(Document(page_content=content, metadata=metadata))
    
    for margin in historical_margins:
        content = f"Category: {margin['category']}\nAverage Margin: {margin['avg_margin']}\nDescription: {margin['description']}"
        metadata = {"type": "historical_margin", "category": margin["category"]}
        documents.append(Document(page_content=content, metadata=metadata))
    
    for decision in pricing_decisions:
        content = f"Product: {decision['product']}\nCategory: {decision['category']}\nCost: {decision['cost']}\nRecommended Price: {decision['recommended_price']}\nMargin: {decision['margin']}\nCompetitor Price: {decision['competitor_price']}\nOutcome: {decision['outcome']}\nReasoning: {decision['reasoning']}"
        metadata = {"type": "pricing_decision", "category": decision["category"], "product": decision["product"]}
        documents.append(Document(page_content=content, metadata=metadata)) 
    
    for guideline in pricing_guidelines:
        content = f"Guideline: {guideline}"
        metadata = {"type": "pricing_guideline"}
        documents.append(Document(page_content=content, metadata=metadata))
    
    return documents

# Create documents
pricing_docs = create_pricing_documents()
print(f"Created {len(pricing_docs)} searchable documents")

Created 20 searchable documents


In [7]:
import chromadb
from langchain_huggingface import HuggingFaceEmbeddings

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

print("Setting up embeddings...")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL_NAME,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

print("Creating vector store...")

db_path = "./db_home"

vectorstore = Chroma.from_documents(
    pricing_docs,
    embeddings,
    persist_directory=db_path
)

print("Vector store created successfully!")
print(f"Indexed {len(pricing_docs)} documents for retrieval")


Setting up embeddings...
Creating vector store...
Vector store created successfully!
Indexed 20 documents for retrieval


## RAG-Enhanced Pricing Agent

Create our enhanced pricing agent that uses retrieval to ground its recommendations.

In [8]:
class RAGPricingAgent:
    def __init__(self, vectorstore, embeddings):
        self.llm = llm
        
        # Store the vector store and embeddings
        self.vectorstore = vectorstore
        self.embeddings = embeddings
        
        self.system_message = SystemMessage(content="""
        You are a RAG-enhanced pricing agent that provides data-driven price recommendations for retail products.
        You have access to a knowledge base of historical pricing decisions, category benchmarks, and pricing guidelines that you can retrieve relevant information from to inform your recommendations.
        """)
        
    def retrieve_relevant_knowledge(self, query: str, k: int = 5):
        """Retrieve relevant pricing knowledge for the query"""
        retriever = self.vectorstore.as_retriever(search_kwargs={"k": k})
        relevant_docs = retriever.invoke(query)
        return relevant_docs
    
    def format_retrieved_context(self, docs):
        """Format retrieved documents into context string"""
        context_parts = []
        for i, doc in enumerate(docs, 1):
            context_parts.append(f"Document {i} (Type: {doc.metadata['type']}):\n{doc.page_content}")
        
        return "\n\n".join(context_parts)
    
    def get_rag_price_recommendation(self, product_name, category, cost_price, 
                                   current_price=None, target_margin=None, 
                                   competitor_price=None, price_elasticity=None):
        """Get price recommendation using RAG"""
        
        search_query = f"Product: {product_name}\nCategory: {category}\nCost Price: {cost_price}\nCurrent Price: {current_price}\nTarget Margin: {target_margin}\nCompetitor Price: {competitor_price}\nPrice Elasticity: {price_elasticity}"  
            
        relevant_docs = self.retrieve_relevant_knowledge(search_query)
        context = self.format_retrieved_context(relevant_docs)
        
        prompt = f"""
You are a pricing agent tasked with recommending an optimal price for the following product:
- Product Name: {product_name}
- Category: {category}   
- Cost Price: {cost_price}
- Current Price: {current_price}
- Target Margin: {target_margin}
- Competitor Price: {competitor_price}
- Price Elasticity: {price_elasticity}
- Retrieved Context: {context}
Provide a price recommendation based on the product details and the retrieved knowledge. Justify your recommendation with reference to the relevant documents and pricing principles.
        """
        
        # Get response from LLM
        messages = [self.system_message, HumanMessage(content=prompt)]
        response = self.llm.invoke(messages)
        
        return response.content

# Initialize the RAG pricing agent
print("Initializing RAG Pricing Agent...")
# rag_agent = RAGPricingAgent(vectorstore, embeddings)
print("RAG Pricing Agent Ready!")

Initializing RAG Pricing Agent...
RAG Pricing Agent Ready!


## Testing the RAG-Enhanced Agent

Test our RAG agent with the assignment example.

In [9]:
rag_agent = RAGPricingAgent(vectorstore, embeddings)

# Test with the assignment example: Puma sneakers
print("Testing Assignment Example: Puma Sneakers")
print("=" * 60)

result = rag_agent.get_rag_price_recommendation(
    product_name="Puma Sneakers",
    category="Footwear", 
    cost_price=1800,  # Note: High cost from assignment
    target_margin=30,
    price_elasticity="High"
)

print(result)

Testing Assignment Example: Puma Sneakers
To determine the optimal price for the Puma Sneakers, we need to consider the provided product details and the insights from the retrieved documents.

1. **Cost Price and Target Margin**: The cost price of the Puma Sneakers is 1800, and the target margin is 30%. To calculate the selling price based on the target margin, we use the formula: Selling Price = Cost Price / (1 - Target Margin). Substituting the given values, we get Selling Price = 1800 / (1 - 0.30) = 1800 / 0.70 = 2571.43.

2. **Price Elasticity**: The price elasticity for the Footwear category is high, as indicated by Document 5 (Type: elasticity_benchmark). This means that small price changes can lead to large changes in demand. Given this high elasticity, aggressive pricing strategies can be effective in capturing market share.

3. **Competitor Pricing and Category Benchmarks**: Although competitor prices for Puma Sneakers are not provided, we can look at similar products and cate

In [10]:
rag_agent = RAGPricingAgent(vectorstore, embeddings)

def compare_rag_vs_baseline(product_name, category, cost_price, target_margin=None, 
                           competitor_price=None, price_elasticity=None):
    """Compare RAG recommendation with baseline prompt-only approach"""
    
    rag_result = rag_agent.get_rag_price_recommendation(
        product_name=product_name,
        category=category,
        cost_price=cost_price,          
        target_margin=target_margin,
        competitor_price=competitor_price,
        price_elasticity=price_elasticity
    )
    
    baseline_prompt = f"""You are a pricing agent that provides price recommendations for retail products based on product details and general pricing principles. 
    Product Name: {product_name}
    Category: {category}
    Cost Price: {cost_price}
    Target Margin: {target_margin}
    Competitor Price: {competitor_price}  
    Price Elasticity: {price_elasticity}
    Provide a price recommendation based on the product details and general pricing principles, without access to any external knowledge or historical data.      
    """
    baseline_response = llm.invoke([SystemMessage(content="You are a pricing agent that provides price recommendations based on product details."), HumanMessage(content=baseline_prompt)]  )
    
    return f"""
    RAG Recommendation:
    {rag_result}    

    Baseline Recommendation:
    {baseline_response.content}
    """

# Test comparison
print("COMPARISON: RAG vs Baseline Approaches")
print("=" * 70)

comparison = compare_rag_vs_baseline(
    "Puma Sneakers",
    "Footwear",
    1800,
    target_margin=30,
    price_elasticity="High"
)

print(comparison)

COMPARISON: RAG vs Baseline Approaches

    RAG Recommendation:
    To determine the optimal price for the Puma Sneakers, we need to consider the provided product details and the insights from the retrieved documents.

1. **Cost Price and Target Margin**: The cost price of the Puma Sneakers is 1800, and the target margin is 30%. To calculate the selling price based on the target margin, we use the formula: Selling Price = Cost Price / (1 - Target Margin). Substituting the given values, we get Selling Price = 1800 / (1 - 0.30) = 1800 / 0.70 = 2571.43.

2. **Price Elasticity**: The price elasticity for the Puma Sneakers is stated as high. According to Document 5 (Type: elasticity_benchmark), high elasticity in the footwear category means that the product is highly sensitive to price changes. This implies that small price drops can lead to large increases in demand.

3. **Competitor Pricing and Category Benchmarks**: Although there is no direct competitor price provided for the Puma Sneak

## Additional Test Cases

Test with various scenarios to see how RAG improves recommendations.

In [11]:
print("Test Case 1: Electronics Category")
print("=" * 40)

electronics_result = rag_agent.get_rag_price_recommendation(
    product_name="Sony 4K TV",
    category="Electronics",
    cost_price=600,
    target_margin=20,
    competitor_price=899,
    price_elasticity="High"
)

print(electronics_result)


Test Case 1: Electronics Category
Based on the product details and the retrieved knowledge, I recommend a price of $839 for the Sony 4K TV. Here's my justification:

1. **Target Margin**: The target margin for the Sony 4K TV is 20%, which means the selling price should be at least $720 (Cost Price + 20% of Cost Price = $600 + 0.20 * $600 = $720).
2. **Competitor Price**: The competitor price is $899, which is higher than our target price. Given the high price elasticity of the product, we can consider pricing below the competitor to drive volume sales.
3. **Pricing Principles**: In high elasticity categories, aggressive pricing can drive volume sales. This is evident from Document 1 and Document 2, where the Samsung Galaxy Smartphone was priced at $499 (11% margin) to gain 15% market share. Similarly, we can apply this principle to the Sony 4K TV.
4. **Category Benchmarks**: Although the retrieved documents are for different products, they provide insights into the electronics category

In [12]:
print("Test Case 2: Luxury Goods Category")
print("=" * 40)

luxury_result = rag_agent.get_rag_price_recommendation(
    product_name="Luxury Watch",
    category="Luxury Goods",
    cost_price=2000,
    target_margin=30,
    competitor_price=3500,
    price_elasticity="Low"
)

print(luxury_result)

Test Case 2: Luxury Goods Category
To determine the optimal price for the Luxury Watch, I will analyze the provided product details and the retrieved context from the knowledge base.

The product details are as follows:
- Product Name: Luxury Watch
- Category: Luxury Goods
- Cost Price: 2000
- Current Price: None
- Target Margin: 30
- Competitor Price: 3500
- Price Elasticity: Low

Given the target margin of 30%, the calculated selling price would be:
Selling Price = Cost Price / (1 - Target Margin)
= 2000 / (1 - 0.30)
= 2000 / 0.70
= 2857.14

However, considering the competitor price of 3500 and the low price elasticity of the luxury goods category, I will analyze the retrieved context to inform my recommendation.

The retrieved documents (1-5) primarily relate to the clothing category, which has medium price elasticity. Although these documents are not directly applicable to the luxury goods category, they demonstrate the importance of competitive pricing in attracting and retaining 

## Exploring the Knowledge Base

See what the RAG system retrieves for different queries.

In [13]:
def explore_knowledge_retrieval(query, k=3):
    """Show what gets retrieved for a given query"""
    print(f"Query: '{query}'")
    print("=" * 50)
    
    docs = rag_agent.retrieve_relevant_knowledge(query, k=k)
    
    for i, doc in enumerate(docs, 1):
        print(f"Document {i} (Type: {doc.metadata['type']}):\n{doc.page_content}\n{'-' * 50}")
    
    return docs

# Test different retrieval queries
explore_knowledge_retrieval("high elasticity pricing strategy")
print("\n" + "=" * 70 + "\n")
explore_knowledge_retrieval("footwear sneakers margin")

Query: 'high elasticity pricing strategy'
Document 1 (Type: pricing_guideline):
Guideline: For high elasticity categories, consider aggressive pricing strategies to drive volume and gain market share.
--------------------------------------------------
Document 2 (Type: pricing_guideline):
Guideline: For high elasticity categories, consider aggressive pricing strategies to drive volume and gain market share.
--------------------------------------------------
Document 3 (Type: pricing_guideline):
Guideline: For medium elasticity categories, maintain competitive pricing to retain customers while ensuring healthy margins.
--------------------------------------------------


Query: 'footwear sneakers margin'
Document 1 (Type: historical_margin):
Category: Footwear
Average Margin: 40-60%
Description: Margins can be substantial, particularly for popular brands and styles, but can vary widely based on market trends and competition.
--------------------------------------------------
Document 2 

[Document(metadata={'type': 'historical_margin', 'category': 'Footwear'}, page_content='Category: Footwear\nAverage Margin: 40-60%\nDescription: Margins can be substantial, particularly for popular brands and styles, but can vary widely based on market trends and competition.'),
 Document(metadata={'category': 'Clothing', 'type': 'historical_margin'}, page_content='Category: Clothing\nAverage Margin: 30-50%\nDescription: Fashion items typically have higher margins, especially for premium brands.'),
 Document(metadata={'category': 'Clothing', 'type': 'historical_margin'}, page_content='Category: Clothing\nAverage Margin: 30-50%\nDescription: Fashion items typically have higher margins, especially for premium brands.')]

## Your Turn: Expand the Knowledge Base

**Exercise 1:** Add 5 more pricing decisions to the knowledge base
**Exercise 2:** Add a new category with appropriate benchmarks
**Exercise 3:** Test how the new data affects recommendations

In [15]:
# Exercise 1: Define new pricing decisions
additional_pricing_decisions = [
    {
        "product": "Adidas Running Shoes",
        "category": "Clothing",
        "cost": 50,
        "recommended_price": 79.99,
        "margin": "60%",
        "competitor_price": 89.99,
        "outcome": "Successful - increased sales by 25%",
        "reasoning": "Competitive pricing in medium elasticity category attracted customers from competitors"
    },
    {
        "product": "Nike Air Max",
        "category": "Footwear",
        "cost": 100,
        "recommended_price": 149.99,
        "margin": "50%",
        "competitor_price": 159.99,
        "outcome": "Successful - increased sales by 20%",
        "reasoning": "Aggressive pricing in high elasticity category captured market share from competitors"
    },
    {
        "product": "Dell XPS Laptop",
        "category": "Electronics",
        "cost": 800,
        "recommended_price": 999,
        "margin": "25%",
        "competitor_price": 1099,
        "outcome": "Successful - maintained market share",
        "reasoning": "Competitive pricing in high elasticity category retained customers while ensuring healthy margins"
    }
]

# Add only items not already in pricing_decisions (check by product name)
existing_products = {d["product"] for d in pricing_decisions}
new_decisions = [d for d in additional_pricing_decisions if d["product"] not in existing_products]
pricing_decisions.extend(new_decisions)
print(f"Added {len(new_decisions)} new pricing decisions (skipped {len(additional_pricing_decisions) - len(new_decisions)} duplicates)")

# Exercise 2: Define and add new category data
new_category_data = {
    "elasticity_benchmark": {
        "category": "Footwear",
        "elasticity": "High",
        "description": "Highly sensitive to price changes, especially for non-essential items, small price drops can lead to large increases in demand."
    },
    "historical_margins": {
        "category": "Footwear",
        "avg_margin": "40-60%",
        "description": "Margins can be substantial, particularly for popular brands and styles, but can vary widely based on market trends and competition."
    }
}

existing_benchmark_cats = {b["category"] for b in elasticity_benchmarks}
if new_category_data["elasticity_benchmark"]["category"] not in existing_benchmark_cats:
    elasticity_benchmarks.append(new_category_data["elasticity_benchmark"])
    print(f"Added elasticity benchmark for: {new_category_data['elasticity_benchmark']['category']}")

existing_margin_cats = {m["category"] for m in historical_margins}
if new_category_data["historical_margins"]["category"] not in existing_margin_cats:
    historical_margins.append(new_category_data["historical_margins"])
    print(f"Added historical margin for: {new_category_data['historical_margins']['category']}")

# Exercise 3: Convert only NEW items to Documents
new_docs = []

for decision in new_decisions:
    content = f"Product: {decision['product']}\nCategory: {decision['category']}\nCost: {decision['cost']}\nRecommended Price: {decision['recommended_price']}\nMargin: {decision['margin']}\nCompetitor Price: {decision['competitor_price']}\nOutcome: {decision['outcome']}\nReasoning: {decision['reasoning']}"
    new_docs.append(Document(page_content=content, metadata={"type": "pricing_decision", "category": decision["category"], "product": decision["product"]}))

if new_category_data["elasticity_benchmark"]["category"] not in existing_benchmark_cats:
    b = new_category_data["elasticity_benchmark"]
    new_docs.append(Document(
        page_content=f"Category: {b['category']}\nElasticity: {b['elasticity']}\nDescription: {b['description']}",
        metadata={"type": "elasticity_benchmark", "category": b["category"]}
    ))

if new_category_data["historical_margins"]["category"] not in existing_margin_cats:
    m = new_category_data["historical_margins"]
    new_docs.append(Document(
        page_content=f"Category: {m['category']}\nAverage Margin: {m['avg_margin']}\nDescription: {m['description']}",
        metadata={"type": "historical_margin", "category": m["category"]}
    ))

print(f"\nCreated {len(new_docs)} new documents to insert")

# Load existing vectorstore and append — preserves all old data
existing_vectorstore = Chroma(persist_directory=db_path, embedding_function=embeddings)
existing_vectorstore.add_documents(new_docs)
print(f"Inserted {len(new_docs)} new documents into existing vectorstore")
print(f"Total collection size: {existing_vectorstore._collection.count()} documents")


Added 0 new pricing decisions (skipped 3 duplicates)
Added elasticity benchmark for: Footwear
Added historical margin for: Footwear

Created 2 new documents to insert


/var/folders/98/sj9l681x6l70gvft58zrcsf00000gn/T/ipykernel_17416/808666112.py:89: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  existing_vectorstore = Chroma(persist_directory=db_path, embedding_function=embeddings)


Inserted 2 new documents into existing vectorstore
Total collection size: 47 documents


In [18]:

# Exercise 3: Test with new Footwear category using the updated vectorstore
rag_agent_updated = RAGPricingAgent(existing_vectorstore, embeddings)

print("Test: New Footwear Category")
print("=" * 40)

footwear_result = rag_agent_updated.get_rag_price_recommendation(
    product_name="Dell XPS Laptop",
    category="Electronics",
    cost_price=80,
    target_margin=50,
    competitor_price=149.99,
    price_elasticity="High"
)

print(footwear_result)


Test: New Footwear Category
To determine the optimal price for the Dell XPS Laptop, we need to consider the provided product details and the insights gained from the retrieved documents.

1. **Cost Price and Target Margin**: The cost price of the Dell XPS Laptop is $80, and the target margin is 50%. This implies that the desired selling price should be $80 (cost) + 50% of $80 (target margin) = $80 + $40 = $120.

2. **Competitor Price**: The competitor price for a similar product is $149.99. Given that the category is Electronics and the price elasticity is high, competitive pricing is crucial to maintain market share.

3. **Price Elasticity**: High price elasticity indicates that small changes in price can lead to significant changes in demand. Therefore, pricing aggressively (lower than competitors) could drive volume, but it must be balanced against the need to maintain a healthy margin.

4. **Retrieved Context**:
   - **Document 1** suggests that competitive pricing in a high elasti